# App Probe Check

Deterministic smoke task. Proves that the **job's run-as identity** can make authenticated requests to the manifest's apps, which the runtime gate needs for `probes` and `min_traces`. Interactive success proves nothing here: the gate notebook runs as the build job's identity, not as you.

For every `app` runtime target it runs the target's `probes` (default `GET /`) with `WorkspaceClient().config.authenticate()` headers, against the URL from `w.apps.get(name)`. Redirects are not followed; a redirect to sign-in means the credential was not accepted.

| Result | Meaning |
|---|---|
| `pass` | Authenticated request accepted, probe expectations met |
| `fail` | Auth rejected (redirect, 401, 403) or a probe expectation failed; the task fails |
| `skipped` | App does not exist yet or is not `RUNNING` (nothing to prove against) |

The identity needs `CAN_USE` on each app.

In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [0]:
import json
import os

dbutils.widgets.text("manifest_path", "", "Manifest path (waves.json)")
dbutils.widgets.text("build_root", "", "Build root (resolves a relative manifest_path)")

manifest_path = dbutils.widgets.get("manifest_path").strip()
build_root = dbutils.widgets.get("build_root").rstrip("/")
if manifest_path and not os.path.isabs(manifest_path):
    manifest_path = os.path.join(build_root, manifest_path)
if not manifest_path:
    raise ValueError("manifest_path is required")

with open(manifest_path) as f:
    manifest = json.load(f)

app_targets = {n: t for n, t in (manifest.get("runtime_targets") or {}).items() if t.get("type") == "app"}
print(f"manifest: {manifest_path}")
print(f"app runtime targets: {sorted(app_targets) or 'none'}")

In [0]:
import sys

from databricks.sdk import WorkspaceClient

# app_probes.py sits next to this notebook (cwd for workspace notebooks).
sys.path.insert(0, os.getcwd())
from app_probes import run_app_probes

w = WorkspaceClient()
identity = w.current_user.me().user_name
print(f"identity: {identity}")

results = {}
for name, tgt in app_targets.items():
    app_name = tgt["name"]
    try:
        app = w.apps.get(app_name)
    except Exception as e:
        results[name] = {"app": app_name, "result": "skipped", "detail": f"app not found ({type(e).__name__})"}
        continue
    state = getattr(getattr(app.app_status, "state", None), "value", None)
    compute = getattr(getattr(app.compute_status, "state", None), "value", None)
    if state != "RUNNING" or compute != "ACTIVE":
        results[name] = {"app": app_name, "result": "skipped", "detail": f"app_status={state} compute={compute}"}
        continue
    probe_results, probs = run_app_probes(w, app_name, tgt.get("probes"))
    results[name] = {"app": app_name, "result": "fail" if probs else "pass", "probes": probe_results, "problems": probs}

for name, r in results.items():
    print(f"  {name} ({r['app']}): {r['result'].upper()} {r.get('detail') or r.get('problems') or ''}")

summary = {
    "identity": identity,
    "status": "skipped" if not results or all(r["result"] == "skipped" for r in results.values())
    else ("fail" if any(r["result"] == "fail" for r in results.values()) else "pass"),
    "targets": results,
}
print(f"app_probe {summary['status']}")
if summary["status"] == "fail":
    # Fail the task so the setup notebook's smoke summary shows it.
    raise RuntimeError("app probes failed as " + identity + ": " + json.dumps(
        {n: r.get("problems") for n, r in results.items() if r["result"] == "fail"}))
dbutils.notebook.exit(json.dumps(summary, default=str))